# 🧹 E-commerce Data Cleaning – Complete Process (Beginner Level)

**Steps we will follow:**
1. Load the data
2. Understand the data
3. Handle missing values
4. Remove duplicates
5. Clean column names
6. Clean text columns
7. Fix data types
8. Validate values (check for wrong data)
9. Find and handle outliers
10. Create new useful columns
11. Final check and save the clean file

👉 Run each cell one by one (**Shift + Enter**).

## Step 1: Import libraries and upload the file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded ✅")

In [ ]:
from google.colab import files

uploaded = files.upload()      # choose ecommerce_dataset_updated.csv

In [ ]:
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("File loaded:", file_name)
df.head()

## Step 2: Understand the data

In [ ]:
print("Rows and columns:", df.shape)
print()
df.info()

In [ ]:
# Numbers summary (min, max, average...)
df.describe()

In [ ]:
# Text columns summary
df.describe(include='object')

**Column meaning**

| Column | Meaning |
|---|---|
| User_ID | Customer id |
| Product_ID | Product id |
| Category | Product category |
| Price (Rs.) | Original price |
| Discount (%) | Discount given |
| Final_Price(Rs.) | Price after discount |
| Payment_Method | How the customer paid |
| Purchase_Date | Date of purchase (day-month-year) |

## Step 3: Handle missing values
Missing values = empty cells. We first count them.

In [ ]:
missing = df.isnull().sum()
print(missing)
print()
print("Total missing values:", missing.sum())
print("Missing % per column:")
print((df.isnull().mean() * 100).round(2))

**How to fix missing values (rules):**
- **Number columns** → fill with the **median**
- **Text columns** → fill with the **most common value (mode)**
- A column with more than **50% missing** → drop the column
- Rows where an important column (like `User_ID`) is missing → drop the row

The code below applies all these rules. If your data has no missing values, nothing will change.

In [ ]:
# 1) Drop columns with more than 50% missing
too_many = df.columns[df.isnull().mean() > 0.5]
df = df.drop(columns=too_many)
print("Dropped columns:", list(too_many))

# 2) Drop rows where User_ID or Product_ID is missing
df = df.dropna(subset=['User_ID', 'Product_ID'])

# 3) Fill number columns with median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# 4) Fill text columns with mode
for col in df.select_dtypes(include=['object', 'string']).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values now:", df.isnull().sum().sum())

## Step 4: Remove duplicate rows

In [ ]:
print("Duplicate rows (all columns same):", df.duplicated().sum())
print("Duplicate User_ID   :", df['User_ID'].duplicated().sum())
print("Duplicate Product_ID:", df['Product_ID'].duplicated().sum())

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Rows removed:", before - len(df))
print("Shape now:", df.shape)

## Step 5: Clean the column names
Names like `Price (Rs.)` are hard to use in code. We make them simple: lowercase, no spaces, no brackets.

In [ ]:
df = df.rename(columns={
    'User_ID': 'user_id',
    'Product_ID': 'product_id',
    'Category': 'category',
    'Price (Rs.)': 'price',
    'Discount (%)': 'discount_pct',
    'Final_Price(Rs.)': 'final_price',
    'Payment_Method': 'payment_method',
    'Purchase_Date': 'purchase_date'
})

print(list(df.columns))

## Step 6: Clean text columns
Remove extra spaces and make spelling/case the same (for example `upi`, `UPI ` and `Upi` should all be one value).

In [ ]:
text_cols = ['user_id', 'product_id', 'category', 'payment_method']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()        # remove spaces at start/end
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)   # many spaces -> one space

df['category'] = df['category'].str.title()          # Home & Kitchen, Books...
df['payment_method'] = df['payment_method'].str.title()
df['payment_method'] = df['payment_method'].replace({'Upi': 'UPI'})

print(df['category'].value_counts())
print()
print(df['payment_method'].value_counts())

## Step 7: Fix data types
`purchase_date` is stored as text. We convert it to a real date so we can find month, weekday etc.

In [ ]:
df['purchase_date'] = pd.to_datetime(df['purchase_date'], format='%d-%m-%Y', errors='coerce')

print("Wrong dates (NaT):", df['purchase_date'].isnull().sum())
print("Date range:", df['purchase_date'].min().date(), "to", df['purchase_date'].max().date())

# Category columns use less memory
df['category'] = df['category'].astype('category')
df['payment_method'] = df['payment_method'].astype('category')

df.dtypes

## Step 8: Validate the values (find wrong data)
Check that the values make sense:
- price must be more than 0
- discount must be between 0 and 100
- `final_price` must equal `price` − discount

In [ ]:
print("Price <= 0           :", (df['price'] <= 0).sum())
print("Discount outside 0-100:", ((df['discount_pct'] < 0) | (df['discount_pct'] > 100)).sum())
print("Final price > price  :", (df['final_price'] > df['price']).sum())
print("Future dates         :", (df['purchase_date'] > pd.Timestamp.today()).sum())

# Recalculate final price and compare
calc_final = (df['price'] * (1 - df['discount_pct'] / 100)).round(2)
mismatch = (abs(calc_final - df['final_price']) > 0.02)
print("Final price mismatch :", mismatch.sum())

In [ ]:
# Fix: remove impossible rows and repair wrong final prices
df = df[(df['price'] > 0) & (df['discount_pct'].between(0, 100))]
df = df.dropna(subset=['purchase_date'])

df['final_price'] = (df['price'] * (1 - df['discount_pct'] / 100)).round(2)

df = df.reset_index(drop=True)
print("Shape now:", df.shape)

## Step 9: Find outliers
A **boxplot** shows outliers as dots outside the box.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.boxplot(y=df['price'], color='skyblue')
plt.title('Price')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['final_price'], color='lightgreen')
plt.title('Final price')

plt.tight_layout()
plt.show()

Now count the outliers using the **IQR rule**.

In [ ]:
def iqr_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (data[column] < lower) | (data[column] > upper)
    return mask, lower, upper

for col in ['price', 'final_price', 'discount_pct']:
    mask, low, up = iqr_outliers(df, col)
    print(f"{col:13s} lower={low:8.2f}  upper={up:8.2f}  outliers={mask.sum()}")

**Remove outliers (optional).**
If the count above is `0`, nothing is removed. Only remove outliers when they are mistakes. Real big purchases should be kept.

In [ ]:
before = len(df)
for col in ['price', 'final_price']:
    mask, low, up = iqr_outliers(df, col)
    df = df[~mask]
df = df.reset_index(drop=True)

print("Rows before:", before)
print("Rows after :", len(df))
print("Rows removed:", before - len(df))

## Step 10: Create new useful columns (Feature Engineering)
From the date and prices we can make new columns that help in analysis.

In [ ]:
df['year'] = df['purchase_date'].dt.year
df['month'] = df['purchase_date'].dt.month
df['month_name'] = df['purchase_date'].dt.month_name()
df['weekday'] = df['purchase_date'].dt.day_name()
df['is_weekend'] = df['purchase_date'].dt.dayofweek >= 5

df['discount_amount'] = (df['price'] - df['final_price']).round(2)

df.head()

## Step 11: Final check of the clean data

In [ ]:
print("Clean data shape:", df.shape)
print("Missing values  :", df.isnull().sum().sum())
print("Duplicate rows  :", df.duplicated().sum())
print()
df.info()

In [ ]:
# Quick graphs to confirm the data looks good
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
df['category'].value_counts().plot(kind='bar', color='orange')
plt.title('Orders by Category')

plt.subplot(1, 3, 2)
df['payment_method'].value_counts().plot(kind='bar', color='seagreen')
plt.title('Orders by Payment Method')

plt.subplot(1, 3, 3)
df.groupby('month')['final_price'].sum().plot(kind='line', marker='o', color='tomato')
plt.title('Total Sales by Month')

plt.tight_layout()
plt.show()

## Step 12: Save the clean file

In [ ]:
df.to_csv('ecommerce_cleaned.csv', index=False)
print("Saved: ecommerce_cleaned.csv ✅")

files.download('ecommerce_cleaned.csv')      # downloads to your computer

## ✅ Done!
Your data is now clean: no missing values, no duplicates, correct data types, valid prices, and new useful columns.